# November 17 — Computational Text Analysis
### Guided, beginner-friendly Colab notebook

> **JOUR 701 | Research Methods in the Age of AI**  
> **Classroom teaching dataset:** The 180 rows in this notebook were **synthetically generated** to resemble the structure of the earlier Block 3 example. The written responses are invented teaching examples; they are **not actual participants’ answers** and must not be reported as real empirical findings.

**How to run:** In Colab, use **Runtime → Run all**, or click the play button beside each code cell from top to bottom. Read the text instructions and answer the reflection prompts.

## Research story: an old experiment, a new kind of outcome
Previously, we studied **Control vs. Personalized** messages using numeric trust and Yes/No clicks. Now imagine adding an open-ended question: **“How did you feel about the message?”** Our goal is to describe what people wrote before trying any sentiment classification.

### Step 1 — Load the course dataset directly from GitHub
**No CSV download or manual upload is necessary.** The dataset is stored in the course GitHub repository's `data/` folder. Run the cell below to load the synthetic practice records automatically. The CSV must first be committed to GitHub, and Colab needs an internet connection.

**Research reminder:** Each row represents a fictional participant. The `open_response` column contains the invented open-ended response.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the Week 14 synthetic dataset directly from our course GitHub repository.
url = "https://raw.githubusercontent.com/KayChansiri/Course_research-methods-in-the-age-of-ai/main/data/week14_synthetic_message_responses.csv"
df = pd.read_csv(url)

print("Rows and columns:", df.shape)
print("Number of participants:", len(df))
display(df.head(5))


### Step 2 — Look at the new text column
**Before running:** Predict what `df["open_response"].head()` will display.

In [ ]:
display(df[["participant_id", "condition", "open_response"]].head(8))
print("Missing text answers:", df["open_response"].isna().sum())

### Step 3 — Count how many words each person wrote
`str.split()` separates a response into words; `len()` counts them. This is a simple descriptive measure, **not** a measure of emotion or quality.

In [ ]:
df["word_count"] = df["open_response"].fillna("").str.split().str.len()
display(df[["condition", "open_response", "word_count"]].head())
print("Average response length:", round(df["word_count"].mean(), 1))

### Step 4 — Compare average lengths between conditions
This is **description**, not yet a test of a hypothesis. Does a higher word count mean the response is more positive?

In [ ]:
length_table = df.groupby("condition")["word_count"].agg(["count", "mean", "median"]).round(1)
display(length_table)
length_table["mean"].plot(kind="bar", title="Average number of words by condition", ylabel="Words", rot=0)
plt.show()

### Step 5 — Prepare text for word-frequency counts
We will make letters lowercase, find word-like sequences, and remove very common **stopwords** (such as “the” and “was”). We are counting **words**, not asking an LLM to interpret them.

**Important:** preprocessing choices can remove meaningful information. For sentiment work, removing words like “not” can be especially harmful.

In [ ]:
import re
from collections import Counter
stopwords = {"the","a","an","and","or","to","of","in","it","this","that","was","is","i","my","me","but","for","with","how","on","very","as","be","at","more","would"}
def simple_tokens(text):
    words = re.findall(r"[a-z]+", str(text).lower())
    return [word for word in words if word not in stopwords]
print(simple_tokens("I liked the post, but I was not convinced."))

### Step 6 — What words appear most often?

In [ ]:
all_words = [w for text in df["open_response"] for w in simple_tokens(text)]
frequent = pd.DataFrame(Counter(all_words).most_common(12), columns=["word", "count"])
display(frequent)
frequent.sort_values("count").plot(kind="barh", x="word", y="count", legend=False, title="Most frequent words")
plt.show()

### Step 7 — Compare words across conditions
A frequent word in one condition might reflect message content rather than sentiment. Always ask **what the word means in context**.

In [ ]:
for condition, group in df.groupby("condition"):
    tokens = [w for text in group["open_response"] for w in simple_tokens(text)]
    print("\n", condition, Counter(tokens).most_common(10))

### Step 8 — Simple keyword coding: does a response mention trust?
This is a **rule-based indicator**. The rule is explicit: look for the words “trust” or “trustworthy.” It does **not** tell us whether the writer trusts the message.

In [ ]:
df["mentions_trust"] = df["open_response"].str.contains(r"\b(?:trust|trustworthy)\b", case=False, regex=True, na=False)
display(pd.crosstab(df["condition"], df["mentions_trust"]))
display(df.loc[df["mentions_trust"], ["open_response", "mentions_trust"]].head(8))

### Step 9 — Connect the new indicator to Block 3 (optional)
We now have **two categorical variables**: `condition` and `mentions_trust`. Which test did we use for that situation? A chi-square test can describe association **if the data and assumptions are appropriate**. Remember: this synthetic dataset is only for technique practice.

In [ ]:
from scipy.stats import chi2_contingency
table = pd.crosstab(df["condition"], df["mentions_trust"])
display(table)
if table.shape == (2, 2) and (table.values > 0).all():
    chi2, p, dof, expected = chi2_contingency(table)
    print(f"Illustrative chi-square: chi2={chi2:.2f}, p={p:.3f}")
else:
    print("Check the table before choosing a statistical test.")

## Think–pair–share (5 minutes)
1. What does a word-frequency chart help us learn?
2. Why might the word **“trust”** appear in both positive and negative responses?
3. Which of these is measured directly: **response length**, **trust mentions**, or **emotion**?
4. Why must we inspect original responses instead of trusting a word count alone?

## Exit ticket
**In one sentence:** What is one text measure you could create from open-ended survey answers, and what is one limitation of that measure?

**Preview:** Next class, we ask whether a computer can assign *positive / neutral / negative* labels to each response.

### Notes / reproducibility
- Do not interpret p-values from invented responses as substantive evidence.
- This notebook uses transparent rules, pandas, and basic Python; it does **not** use an LLM.
- The `condition` and traditional outcome columns mirror the **structure** described in the earlier Week 13 slides; the current rows and text are fabricated.
- Code and activities can be modified without changing the conceptual progression.